# Notebook 02 : Parsing des données et construction des DataFrames

## Objectif
Ce notebook transforme les fichiers XML bruts en **DataFrames Pandas** exploitables :
1. **Parsing des députés** : extraction de l'identité et de l'appartenance politique de chaque député
2. **Parsing des débats** : extraction de chaque prise de parole avec son texte, sa date et l'identité de l'orateur
3. **Fusion** : on associe à chaque prise de parole le parti politique de l'orateur
4. **Filtrage VSS** : on ne conserve que les prises de parole qui mentionnent des mots-clés VSS, en excluant les faux positifs
5. **Ajout des blocs idéologiques** et du texte nettoyé

## Entrées
- `df_deputes.csv` (ou les fichiers XML des acteurs/organes si première exécution)
- Fichiers XML dans les dossiers `sorted/`

## Sorties
- `df_deputes.csv` : DataFrame des députés et partis
- `df_global.pkl` : Toutes les prises de parole parsées (avec parti)
- `df_vss_propre.pkl` : Prises de parole VSS filtrées, avec blocs et texte nettoyé

## 1. Imports et configuration

In [17]:
from config import *
from lxml import etree
import pandas as pd
import numpy as np
import os, re
from tqdm import tqdm

import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import FrenchStemmer
nltk.download('stopwords', quiet=True)

NAMESPACES = {'ns': 'http://schemas.assemblee-nationale.fr/referentiel'}
ADRESSE = "{http://schemas.assemblee-nationale.fr/referentiel}"

stemmer = FrenchStemmer()

print(" Imports terminés.")

 Imports terminés.


## 2. Parsing des députés et partis politiques

On extrait depuis les fichiers XML des acteurs et organes :
- **Les partis politiques** (organes de type `PARPOL`) : identifiant et nom
- **Les députés** : identifiant, nom, prénom, parti et date de début de mandat

Un député peut apparaître plusieurs fois s'il a changé de parti au cours de sa carrière.


In [18]:
# Suppression du cache pour forcer le recalcul
for f in [CHEMIN_DF_DEPUTES]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Cache supprime : {f}")


if os.path.exists(CHEMIN_DF_DEPUTES):
    df_deputes = pd.read_csv(CHEMIN_DF_DEPUTES)
    df_deputes["debut"] = pd.to_datetime(df_deputes["debut"], errors='coerce')
    df_deputes["parti"] = df_deputes["parti"].astype(str)
    print(f" df_deputes chargé depuis le cache : {len(df_deputes)} entrées.")
    print(f"   Colonnes : {df_deputes.columns.tolist()}")

else:
    print(" Construction de df_deputes depuis les fichiers XML...")

    # --- Parsing des partis ---
    data_partis = []
    if os.path.exists(DOSSIER_PARTIS):
        for fichier in os.listdir(DOSSIER_PARTIS):
            if not fichier.endswith('.xml'):
                continue
            tree = etree.parse(os.path.join(DOSSIER_PARTIS, fichier))
            root = tree.getroot()
            data_partis.append({
                "parti": root.xpath('//ns:uid', namespaces=NAMESPACES)[0].text,
                "nom_parti": root.xpath('//ns:libelle', namespaces=NAMESPACES)[0].text
            })
    df_partis = pd.DataFrame(data_partis)
    df_partis["parti"] = df_partis["parti"].astype(str)
    print(f"   {len(df_partis)} partis trouvés.")

    # --- Parsing des députés ---
    data_deputes = []
    if os.path.exists(DOSSIER_DEPUTES):
        for fichier in os.listdir(DOSSIER_DEPUTES):
            if not fichier.endswith('.xml'):
                continue
            try:
                tree = etree.parse(os.path.join(DOSSIER_DEPUTES, fichier))
                root = tree.getroot()
                id_acteur = root.xpath('//ns:uid', namespaces=NAMESPACES)[0].text
                nom = root.xpath('//ns:etatCivil/ns:ident/ns:nom', namespaces=NAMESPACES)[0].text
                prenom = root.xpath('//ns:etatCivil/ns:ident/ns:prenom', namespaces=NAMESPACES)[0].text

                for mandat in root.xpath('//ns:mandats/ns:mandat', namespaces=NAMESPACES):
                    type_organe = mandat.xpath('./ns:typeOrgane/text()', namespaces=NAMESPACES)
                    if type_organe and type_organe[0] == "PARPOL":
                        organe_ref = mandat.xpath('./ns:organes/ns:organeRef/text()', namespaces=NAMESPACES)
                        date_debut = mandat.xpath('./ns:dateDebut/text()', namespaces=NAMESPACES)
                        if organe_ref and date_debut:
                            data_deputes.append({
                                "id_acteur": id_acteur, "nom": nom, "prenom": prenom,
                                "parti": organe_ref[0], "debut": date_debut[0]
                            })
            except Exception:
                continue

    df_deputes = pd.DataFrame(data_deputes)
    df_deputes["parti"] = df_deputes["parti"].astype(str)
    df_deputes["debut"] = pd.to_datetime(df_deputes["debut"], errors='coerce')
    df_deputes = df_deputes.merge(df_partis, on="parti", how="left")

    # Sauvegarde
    df_deputes.to_csv(CHEMIN_DF_DEPUTES, index=False)
    print(f"   {len(df_deputes)} entrées député-parti trouvées.")
    print(f" Sauvegardé dans {CHEMIN_DF_DEPUTES}")

 df_deputes chargé depuis le cache : 3039 entrées.
   Colonnes : ['Unnamed: 0', 'id_acteur', 'nom', 'prenom', 'parti', 'debut', 'nom_parti']


## 3. Parsing des débats

La fonction `parsing_debats` extrait de chaque fichier XML de séance :
- L'**identifiant** de la séance, la **date** et la **législature**
- Chaque **prise de parole** : texte, identifiant du député, code grammaire

Les fichiers XML de l'Assemblée ont une structure imbriquée avec des `<point>` pouvant contenir des `<interExtraction>` (interventions) et des `<paragraphe>`. On explore toutes les profondeurs possibles grâce à XPath récursif.

In [19]:
def parsing_debats(fichier):
    """
    Parse un fichier XML de séance et retourne un DataFrame des prises de parole.

    Args:
        fichier (str): chemin complet vers le fichier XML

    Returns:
        pd.DataFrame: une ligne par prise de parole, avec colonnes
                      [seanceRef, ordre_absolu_seance, date, legislature,
                       id_acteur, code_grammaire, texte]
    """
    try:
        tree = etree.parse(fichier)
        root = tree.getroot()
        data_list = []

        # Métadonnées communes à toute la séance
        uid_seance = root.findall(ADRESSE + "uid")[0].text
        date_brute = root.findall(ADRESSE + "metadonnees/" + ADRESSE + "dateSeance")[0].text[:8]
        date_seance = pd.to_datetime(date_brute)
        legis = root.findall(ADRESSE + "metadonnees/" + ADRESSE + "legislature")[0].text

        # Recherche des noeuds paragraphe et interExtraction à toutes les profondeurs
        # On utilise XPath récursif pour ne rien manquer
        nodes_inter = root.xpath('.//ns:interExtraction', namespaces=NAMESPACES)
        nodes_para = root.xpath(
            './/ns:contenu//ns:paragraphe[not(ancestor::ns:interExtraction)]',
            namespaces=NAMESPACES
        )

        for node in nodes_inter + nodes_para:
            # Extraction du texte
            parts = []
            for j in node.findall(ADRESSE + "texte"):
                if j.text:
                    parts.append(j.text)
                for child in j:
                    if child.text:
                        parts.append(child.text)
                    if child.tail:
                        parts.append(child.tail)

            texte_final = "".join(parts).replace("\xa0", " ").strip()
            if not texte_final:
                continue

            data_list.append({
                "seanceRef": uid_seance,
                "ordre_absolu_seance": node.get("ordre_absolu_seance"),
                "date": date_seance,
                "legislature": legis,
                "id_acteur": node.get("id_acteur"),
                "code_grammaire": node.get("code_grammaire"),
                "texte": texte_final
            })

        df = pd.DataFrame(data_list)
        if not df.empty:
            df["ordre_absolu_seance"] = pd.to_numeric(df["ordre_absolu_seance"], errors='coerce')
            df = df.sort_values(by="ordre_absolu_seance")
        return df

    except Exception as e:
        return pd.DataFrame()

## 4. Jointure des partis politiques

Pour chaque prise de parole, on cherche le parti du député **au moment de la séance**. Un député ayant pu changer de parti, on sélectionne le mandat le plus récent antérieur à la date de la séance.

In [20]:
def ajouter_partis(df_debats, df_deputes):
    """
    Joint le nom du parti politique à chaque prise de parole.

    Sélectionne le mandat partisan le plus récent AVANT la date de la séance.
    """
    if df_debats.empty:
        return df_debats

    date_seance = pd.to_datetime(df_debats.iloc[0]["date"])

    # Filtrer les députés présents dans cette séance
    concernes = df_deputes[df_deputes["id_acteur"].isin(df_debats["id_acteur"])].copy()
    concernes["debut"] = pd.to_datetime(concernes["debut"], errors='coerce')

    # Ne garder que les mandats commencés AVANT la séance
    concernes = concernes[concernes["debut"] <= date_seance]

    # Garder le mandat le plus récent pour chaque député
    concernes = (concernes
                 .sort_values("debut", ascending=False)
                 .drop_duplicates(subset="id_acteur", keep="first"))

    # Jointure
    cols_merge = ["id_acteur", "parti", "nom_parti"]
    if "nom" in concernes.columns:
        cols_merge.extend(["nom", "prenom"])

    return df_debats.merge(concernes[cols_merge], on="id_acteur", how="left")

## 5. Construction du DataFrame global

On parse toutes les séances triées (dossiers `sorted/`) et on fusionne avec les partis.


In [21]:
# Suppression du cache pour forcer le recalcul
for f in [CHEMIN_DF_GLOBAL]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Cache supprime : {f}")

if os.path.exists(CHEMIN_DF_GLOBAL):
    df_global = pd.read_pickle(CHEMIN_DF_GLOBAL)
    print(f" df_global chargé depuis le cache : {len(df_global)} prises de parole.")

else:
    print("Parsing de toutes les séances...")
    all_seances = []

    for leg, chemin in DOSSIER_SORTED.items():
        if not os.path.exists(chemin):
            print(f"   Dossier {chemin} introuvable.")
            continue

        fichiers = [f for f in os.listdir(chemin) if f.endswith('.xml')]
        print(f"   {leg.upper()} : {len(fichiers)} fichiers")

        for fichier in tqdm(fichiers, desc=f"Parsing {leg}"):
            try:
                df_temp = parsing_debats(os.path.join(chemin, fichier))
                if df_temp.empty:
                    continue
                df_temp = ajouter_partis(df_temp, df_deputes)
                all_seances.append(df_temp)
            except Exception as e:
                print(f"   Erreur sur {fichier} : {e}")
                continue

    if all_seances:
        df_global = pd.concat(all_seances, ignore_index=True)
        df_global.to_pickle(CHEMIN_DF_GLOBAL)
        print(f"\n{len(df_global)} prises de parole extraites.")
        print(f"Sauvegardé dans {CHEMIN_DF_GLOBAL}")
    else:
        df_global = pd.DataFrame()
        print("Aucune séance n'a pu être parsée.")

 df_global chargé depuis le cache : 220806 prises de parole.


## 6. Filtrage des prises de parole VSS

On applique deux filtres successifs :
1. **Filtre positif** : on ne garde que les prises de parole contenant au moins un mot-cle de la liste `A_TESTER`
2. **Filtre negatif contextuel** : on retire les faux positifs, mais uniquement lorsque le mot d'exclusion apparait dans la **meme phrase** qu'un mot-cle VSS. Cela evite de rejeter des discours authentiquement VSS qui mentionnent accessoirement un terme d'exclusion dans un autre passage (ex : un debat sur les violences conjugales qui mentionne aussi le budget).

> **Choix methodologique** : un filtre negatif applique au niveau de la prise de parole entiere (comme dans la version precedente) introduit un biais de selection systematique : il supprime preferentiellement les discours longs et thematiquement riches, qui ont plus de chances de contenir un mot d'exclusion par hasard.

On ajoute ensuite la colonne `bloc` (regroupement ideologique) et le texte nettoye (stemming, suppression des stop words).

In [22]:
# Suppression des caches pour forcer le recalcul
import glob as _glob
for pattern in [CHEMIN_DF_VSS_PROPRE, "/home/onyxia/work/projet_eco_socio/df_vss_propre.pkl"]:
    if os.path.exists(pattern):
        os.remove(pattern)
        print(f"Cache supprime : {pattern}")

# ==========================================================================
# CACHE : Si df_vss_propre.pkl existe déjà, on le charge
# ==========================================================================

chemin_vss_racine = "/home/onyxia/work/projet_eco_socio/df_vss_propre.pkl"
if os.path.exists(chemin_vss_racine):
    CHEMIN_DF_VSS_PROPRE = chemin_vss_racine

if os.path.exists(CHEMIN_DF_VSS_PROPRE):
    df_vss = pd.read_pickle(CHEMIN_DF_VSS_PROPRE)
    print(f" df_vss chargé depuis le cache : {len(df_vss)} prises de parole VSS.")
    print(f"   Répartition par bloc :")
    print(df_vss['bloc'].value_counts().to_string())

else:
    print(" Filtrage des prises de parole VSS...")

    # --- Filtre Positif ---
    pattern_positif = '|'.join([re.escape(mot) for mot in A_TESTER])
    df_vss = df_global[
        df_global['texte'].str.contains(pattern_positif, case=False, na=False)
    ].copy()

    nb_avant = len(df_vss)

    # --- Filtre Negatif (contextuel, par phrase) ---
    # Correction : on ne supprime plus une prise de parole entiere si un mot
    # d'exclusion apparait quelque part dans le texte. On verifie que le mot
    # d'exclusion et le mot VSS n'apparaissent pas dans la MEME phrase.
    # Cela evite de perdre des discours VSS authentiques qui mentionnent
    # accessoirement un terme d'exclusion dans un autre passage.
    pattern_negatif = re.compile(
        '|'.join([re.escape(mot) for mot in MOTS_EXCLUSION]), re.IGNORECASE
    )
    pattern_positif_check = re.compile(
        '|'.join([re.escape(mot) for mot in A_TESTER]), re.IGNORECASE
    )

    def est_faux_positif(texte):
        """Verifie si TOUTES les mentions VSS du texte cooccurrent avec un mot d'exclusion."""
        phrases = re.split(r'[.!?;]+', str(texte))
        phrases_vss = [p for p in phrases if pattern_positif_check.search(p)]
        if not phrases_vss:
            return True
        # Si au moins une phrase VSS ne contient pas de mot d'exclusion, on garde
        for p in phrases_vss:
            if not pattern_negatif.search(p):
                return False
        return True

    nb_avant_neg = len(df_vss)
    df_vss = df_vss[~df_vss['texte'].apply(est_faux_positif)]

    nb_rejetes = nb_avant - len(df_vss)
    print(f"   {len(df_vss)} prises de parole VSS retenues")
    print(f"   {nb_rejetes} faux positifs supprimés")

    # --- Ajout des blocs idéologiques ---
    df_vss['bloc'] = df_vss['nom_parti'].apply(regrouper_blocs_ideologiques)
    df_vss = df_vss.dropna(subset=['bloc'])

    # --- Nettoyage textuel (stemming + suppression stopwords) ---
    stop_words = stopwords.words('french')
    bruit_parlementaire = [
        "loi", "droit", "état", "amend", "articl", "text", "group",
        "person", "social", "polit", "publi", "monsieur", "madame",
        "président", "ministre", "député", "collègue", "assemblée",
        "proposit", "disposit", "cadre", "mesur", "moyen", "question",
        "rapport", "commission", "gouvern", "souhait", "permettr",
        "faut", "doit", "peut", "bien", "fait", "être", "avoir",
        "plus", "tout", "tous", "faire", "encore", "donc", "vraiment",
    ]
    stop_words.extend(bruit_parlementaire)

    def pre_processing(texte):
        texte = re.sub(r'\W+', ' ', str(texte).lower())
        mots = texte.split()
        mots_nettoyes = [
            stemmer.stem(m) for m in mots
            if m not in stop_words
            and not any(vss in m for vss in A_TESTER)
            and len(m) > 2
        ]
        return " ".join(mots_nettoyes)

    print("   Nettoyage textuel en cours...")
    df_vss['texte_clean'] = df_vss['texte'].apply(pre_processing)

    # --- Sauvegarde ---
    df_vss.to_pickle(CHEMIN_DF_VSS_PROPRE)
    print(f"\n{len(df_vss)} prises de parole VSS pures.")
    print(f"Sauvegardé dans {CHEMIN_DF_VSS_PROPRE}")
    print(f"\n   Répartition par bloc :")
    print(df_vss['bloc'].value_counts().to_string())

 df_vss chargé depuis le cache : 7815 prises de parole VSS.
   Répartition par bloc :
bloc
Centre                   2815
Gauche Radicale          1689
Droite Traditionnelle    1470
Gauche Modérée           1115
Extrême Droite            726


## 7. Vérification rapide

In [23]:
# Échantillon aléatoire pour vérifier la qualité du parsing
print("Exemple de prise de parole VSS :\n")
sample = df_vss.sample(1).iloc[0]
print(f"   Date : {sample['date']}")
print(f"   Parti : {sample.get('nom_parti', '?')}")
print(f"   Bloc : {sample['bloc']}")
print(f"   Texte : {sample['texte'][:300]}...")

Exemple de prise de parole VSS :

   Date : 2021-12-02 00:00:00
   Parti : Les Républicains
   Bloc : Droite Traditionnelle
   Texte : Ils nous disent qu’ils ne comprennent pas l’entêtement borné du Gouvernement à dire non, à refuser, à bloquer, à empêcher une loi qui, pourtant, est voulue par tous. Il est temps de remettre en cause le statut – inique – de minimum social de l’AAH, et de considérer cette allocation avant tout comme ...
